# 03 — Evaluation & Benchmarking (Colab, GPU)

**Owner:** Temirlan (SCRUM-16). Runs the full benchmark: YOLO11n (baseline detector),
Faster R-CNN (challenger detector), and the CNN classification floor — all scored by the
**same** `src/eval` pipeline on the canonical held-out test split.

**Run on Colab with a GPU runtime** (Runtime → Change runtime type → GPU). Faster R-CNN
training and the hyperparameter sweep are infeasible on CPU.

All heavy logic lives in `src/` — these cells stay thin per the repo convention.

## 1. Setup — clone repo, install deps, get data

In [ ]:
# Colab setup. Skip the clone if running inside an existing checkout.
import os, sys
if not os.path.exists('src'):
    !git clone https://github.com/neuroarcane/dental-cavity-detector.git
    %cd dental-cavity-detector
!pip install -q -r requirements.txt
sys.path.insert(0, os.getcwd())

# Dataset: on Colab the two source zips are copied+extracted from Drive by the data
# pipeline. See data/README.md / src/data/config.py (COLAB_DRIVE_PROJECT_FOLDER).
from google.colab import drive
drive.mount('/content/drive')

## 2. Reproduce the canonical test split (dedup + source-stratified 70/15/15, seed 42)

In [ ]:
from src.data.prepare_dataset import merge_datasets, print_merge_summary
from src.data.split import remove_exact_duplicates, stratified_resplit
from src.data.config import DATA_PROCESSED

if not (DATA_PROCESSED / 'test' / 'images').exists():
    print_merge_summary(merge_datasets())
    print(remove_exact_duplicates())
    print(stratified_resplit(random_state=42))
else:
    print('data/processed already built; skipping merge/split')
# NOTE: do NOT oversample here — valid/test must stay at the true distribution.

## 3. Download trained model weights from Hugging Face

In [ ]:
import urllib.request, os
os.makedirs('models', exist_ok=True)
BASE = 'https://huggingface.co/aparnamohankumar/dental-cavity-detector/resolve/main/'
for f in ['yolo11_baseline_best.pt', 'cnn_baseline.keras']:
    dst = f'models/{f}'
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE + f, dst)
    print('ready:', dst, round(os.path.getsize(dst)/1e6, 2), 'MB')

## 4. YOLO11n — evaluate (baseline detector)

Predict at low confidence so mAP integrates the full PR curve; NMS IoU 0.7 per the metrics
spec. `build_detection_report` writes metrics JSON + confusion-matrix / PR / per-class-AP
plots; `render_samples` writes PHI-safe GT-vs-pred panels.

In [ ]:
import time
from pathlib import Path
from ultralytics import YOLO
from src.config import CLASS_NAMES
from src.data.config import DATA_PROCESSED
from src.eval import build_detection_report, load_ground_truth, render_samples
from src.eval.formats import from_yolo_result

test_images = Path(DATA_PROCESSED) / 'test' / 'images'
gt = load_ground_truth('test')

ym = YOLO('models/yolo11_baseline_best.pt')
yolo_det, speeds = {}, []
t0 = time.time()
for r in ym.predict(source=str(test_images), stream=True, verbose=False, conf=0.001, iou=0.7, imgsz=640):
    yolo_det[Path(r.path).name] = from_yolo_result(r)
    speeds.append(r.speed['inference'])
yolo_eff = {'params': sum(p.numel() for p in ym.model.parameters()),
            'size_mb': round(Path('models/yolo11_baseline_best.pt').stat().st_size/1e6, 2),
            'avg_inference_ms': round(sum(speeds)/len(speeds), 1)}
yolo_metrics = build_detection_report(yolo_det, gt, CLASS_NAMES, 'reports/yolo', 'YOLO11n', efficiency=yolo_eff)
render_samples(yolo_det, gt, test_images, CLASS_NAMES, 'reports/yolo/samples', n_good=5, n_failure=3)
print('YOLO mAP@0.5=%.4f  mAP@0.5:0.95=%.4f  (expect ~0.746 / ~0.447)' % (yolo_metrics['map50'], yolo_metrics['map50_95']))

## 5. Faster R-CNN — train the challenger (GPU)

Full training. Match YOLO's budget as closely as the frameworks allow, and run several
seeds (per the benchmark methodology) so a difference is separable from noise. Start with
one seed to validate, then loop seeds.

In [ ]:
from src.models.faster_rcnn import train_faster_rcnn
import torch
print('CUDA available:', torch.cuda.is_available())

# One full run first (raise epochs to match YOLO's training budget; keep batch modest for VRAM).
# save_path persists the checkpoint (gitignored) so we can reload it for eval and share it.
frcnn, history = train_faster_rcnn(epochs=20, batch_size=4, lr=0.005, num_workers=2,
                                   device='cuda', save_path='models/faster_rcnn_best.pt')
print('per-epoch loss:', [round(x, 4) for x in history['epoch_loss']])

In [ ]:
# Evaluate the trained Faster R-CNN with the SAME pipeline as YOLO -> directly comparable.
import os
from src.eval.predict import predict_torchvision_split

frcnn_det = predict_torchvision_split(frcnn, 'test', device='cuda', score_threshold=0.0)
frcnn_eff = {'params': sum(p.numel() for p in frcnn.parameters()),
             'size_mb': round(os.path.getsize('models/faster_rcnn_best.pt')/1e6, 2),
             'avg_inference_ms': None}
frcnn_metrics = build_detection_report(frcnn_det, gt, CLASS_NAMES, 'reports/frcnn', 'FasterRCNN', efficiency=frcnn_eff)
render_samples(frcnn_det, gt, test_images, CLASS_NAMES, 'reports/frcnn/samples', n_good=5, n_failure=3)
print('Faster R-CNN mAP@0.5=%.4f  mAP@0.5:0.95=%.4f' % (frcnn_metrics['map50'], frcnn_metrics['map50_95']))

## 6. CNN classification floor — bridged comparison

The CNN has no boxes, so it is scored with classification metrics, and YOLO is 'downgraded'
to one label/image (priority reduction) for an equal-terms comparison. Localization is
intentionally dropped here — it is reported separately via mAP above.

In [ ]:
import numpy as np, cv2, tensorflow as tf
from src.eval import compute_classification_metrics, image_labels_from_detections, image_labels_from_ground_truth

cnn = tf.keras.models.load_model('models/cnn_baseline.keras')

# CNN predictions: one label per test image (128x128 RGB input).
cnn_pred = {}
for img_path in sorted((Path(DATA_PROCESSED)/'test'/'images').iterdir()):
    im = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
    im = cv2.resize(im, (128, 128)).astype('float32') / 255.0
    cnn_pred[img_path.name] = int(np.argmax(cnn.predict(im[None], verbose=0)[0]))

y_true = image_labels_from_ground_truth(gt, CLASS_NAMES)          # priority-reduced GT
yolo_as_clf = image_labels_from_detections(yolo_det, CLASS_NAMES, strategy='priority')

cnn_m  = compute_classification_metrics(y_true, cnn_pred, CLASS_NAMES)
yolo_m = compute_classification_metrics(y_true, yolo_as_clf, CLASS_NAMES)
print('CNN  accuracy=%.3f  macroF1=%.3f' % (cnn_m['accuracy'], cnn_m['macro']['f1']))
print('YOLO (downgraded) accuracy=%.3f  macroF1=%.3f' % (yolo_m['accuracy'], yolo_m['macro']['f1']))

## 7. Hyperparameter sweep (Cavity-recall objective)

Each trial fine-tunes YOLO, so keep `epochs` modest and the trial count small — a sweep
ranks configs relative to each other; re-train the winner at full budget afterwards.

In [ ]:
from functools import partial
from src.tuning import run_random_sweep, results_to_df, train_yolo, RANDOM_SEARCH_SPACE

data_yaml = str(Path(DATA_PROCESSED) / 'data.yaml')
trainer = partial(train_yolo, weights='yolo11n.pt', data_yaml=data_yaml,
                  epochs=15, project='runs/sweep', name='trial')

# Start small (e.g. 6 trials) to gauge GPU cost, then widen.
trials = run_random_sweep(trainer, RANDOM_SEARCH_SPACE, n_trials=6, seed=42)
df = results_to_df(trials, sort_by='map50_95')
df.to_csv('reports/sweep_results.csv', index=False)
df.head(10)

## 8. Combined benchmark table

In [ ]:
import json, pandas as pd
from pathlib import Path

rows = []
for name, path in [('YOLO11n', 'reports/yolo/YOLO11n_metrics.json'),
                   ('FasterRCNN', 'reports/frcnn/FasterRCNN_metrics.json')]:
    if Path(path).exists():
        m = json.load(open(path))
        rows.append({'model': name, 'mAP@0.5': round(m['map50'], 3), 'mAP@0.5:0.95': round(m['map50_95'], 3),
                     'macro_P': round(m['precision_recall_f1']['macro']['precision'], 3),
                     'macro_R': round(m['precision_recall_f1']['macro']['recall'], 3),
                     'macro_F1': round(m['precision_recall_f1']['macro']['f1'], 3),
                     'params': m['efficiency'].get('params'), 'size_mb': m['efficiency'].get('size_mb')})
pd.DataFrame(rows)